# PINN VFM master guide

Physics-informed virtual flow meter for an ESP-lifted well. The network is a dense residual encoder-decoder LSTM. It predicts the change in normalized bottom-hole pressure (`Pbh`) and wellhead pressure (`Pwh`) and adds that change to the last measured pressures. Flow rate `q` is the third head output and is not fed back during rollout.

Run this notebook from the `pinn` repository root.

## When to use each PINN mode

| pinn_mode | When to use it |
| --- | --- |
| `off` | Pure data-driven baseline when well physical parameters (rho, PI) are unknown. |
| `on` | Standard Virtual Flow Metering (VFM). Use when flow rate q is unmeasured and must be soft-sensed via pressure residuals. |
| `all` | Use when boundary pressure measurements are noisy. Enforces physics across all state boundaries. |
| `loss2` | Use when testing choke valve disturbance stability. |

## When to use each optimizer

| Optimizer | When to use it |
| --- | --- |
| Adam | Initial exploration and training from scratch (fast global descent). |
| L-BFGS | Use immediately after Adam for second-order precision to enforce stiff ODE residuals (<1e-5) and calibrate rho and PI. |


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "pinn_BCS.py").is_file():
    raise FileNotFoundError("Run this notebook from the pinn repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"torch {torch.__version__}  device {device}")


In [ ]:
NPZ_PATH = Path("./dataset/BCS_data_train_limitado_f_zc_pm_pr_ident.npz")
raw = np.load(NPZ_PATH)
pbh_bar = np.asarray(raw["x1"], dtype=float).reshape(-1) / 1e5
pwh_bar = np.asarray(raw["x2"], dtype=float).reshape(-1) / 1e5
q_m3h = np.asarray(raw["x3"], dtype=float).reshape(-1) * 3600.0
t = np.asarray(raw["t"]).reshape(-1)
print(f"file {NPZ_PATH}")
print(f"samples {t.size}   t {int(t[0])} .. {int(t[-1])}")
print(f"Pbh bar   min {pbh_bar.min():.3f}  max {pbh_bar.max():.3f}  mean {pbh_bar.mean():.3f}")
print(f"Pwh bar   min {pwh_bar.min():.3f}  max {pwh_bar.max():.3f}  mean {pwh_bar.mean():.3f}")
print(f"q m3/h    min {q_m3h.min():.3f}  max {q_m3h.max():.3f}  mean {q_m3h.mean():.3f}")
print(f"U shape {tuple(raw['U'].shape)}  columns f, zc, pm, pr")


In [ ]:
from data.BuildDataset import BuildingDataset

N_IN = 20
N_OUT = 1
BATCH = 500
split_point = int(0.7 * len(raw["x1"]))
ds = BuildingDataset(N_IN, N_OUT, raw, split_point, batch_size=BATCH, seed=SEED)
n_windows = int(ds.train_X.shape[0] + ds.test_X.shape[0])
print(f"split_point {split_point}  ({split_point / len(raw['x1']):.0%} of the raw record)")
print(f"windows {n_windows}  train {tuple(ds.train_X.shape)}  test {tuple(ds.test_X.shape)}")
print(f"train target pressures {tuple(ds.train_y.shape)}  test target Pbh,Pwh,q {tuple(ds.test_y.shape)}")
print(f"sliding window N_in={N_IN} N_out={N_OUT}")


In [ ]:
from data.Logger import Logger
from data.pinn_BCS import pinn_vfm

np.random.seed(SEED)
torch.manual_seed(SEED)
logger = Logger(frequency=1)
pinn = pinn_vfm(
    10,
    logger,
    var=[0.9, 0.9],
    pinn_mode="on",
    inputs=6,
    n_steps_in=ds.n_steps_in,
    n_steps_out=ds.n_steps_out,
    parameters=ds.parameters,
    lr=0.01,
    device=str(device),
)
print(pinn.summary_model())
print(f"pinn_mode {pinn.pinn_mode_name}  device {pinn.device}")


In [ ]:
from data.utils import Struct, save_model_files

# Defaults match main_train_adam.py (200) and main_train_lbfgs.py (100).
# Logger(frequency=1) prints the total, data, physics, and test loss every Adam epoch.
ADAM_EPOCHS = int(os.environ.get("PINN_ADAM_EPOCHS", "200"))
LBFGS_ITERS = int(os.environ.get("PINN_LBFGS_ITERS", "100"))
ADAM_DIR = os.environ.get("PINN_ADAM_DIR", "model_adam_200")
LBFGS_DIR = os.environ.get("PINN_LBFGS_DIR", "model_adam_lbfgs")

pinn.lamb_l1 = 1.0
pinn.lamb_l2 = 1.0
pinn.lamb_l3 = 1.0
loss_history, trainstate, var_history = pinn.fit(ds.adam_dataset(), tf_epochs=ADAM_EPOCHS)
save_model_files(
    ADAM_DIR,
    {"Loss": loss_history, "trainstate": trainstate, "vartrain": var_history},
    pinn,
)
print(
    f"Adam  epochs {ADAM_EPOCHS}  total {float(trainstate.loss_train):.3e}  "
    f"data {float(trainstate.loss_train_bc):.3e}  physics {float(trainstate.loss_train_f):.3e}  "
    f"test {float(trainstate.loss_test):.3e}"
)

pinn.lamb_l1 = 2.0
pinn.lamb_l2 = 1.9
pinn.lamb_l3 = 0.01
nt = Struct()
nt.maxIter = LBFGS_ITERS
nt.nCorrection = 50
nt.tol = 1e-5
nt.tolfun = 1e-5
loss_history, trainstate, var_history = pinn.fit_LBFGS(ds.lbfgs_dataset(), nt)
save_model_files(
    LBFGS_DIR,
    {"Loss": loss_history, "trainstate": trainstate, "vartrain": var_history},
    pinn,
)
print(
    f"L-BFGS  iterations {LBFGS_ITERS}  total {float(trainstate.loss_train):.3e}  "
    f"data {float(trainstate.loss_train_bc):.3e}  physics {float(trainstate.loss_train_f):.3e}  "
    f"test {float(trainstate.loss_test):.3e}  "
    f"rho {float(pinn.rho.detach().cpu()) * 950:.2f}  "
    f"PI {float(pinn.PI.detach().cpu()) * 2.32e-9:.3e}"
)
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(loss_history.steps, loss_history.loss_train, label="train")
ax.semilogy(loss_history.steps, loss_history.loss_test, label="test")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
model = pinn.u_model
model.eval()
test_x = torch.as_tensor(ds.test_X, dtype=torch.float32, device=pinn.device)
test_y = np.asarray(ds.test_y, dtype=np.float32)
chunks = []
with torch.inference_mode():
    for start in range(0, int(test_x.shape[0]), 1024):
        chunks.append(model(test_x[start:start + 1024]).detach().cpu().numpy())
pred = np.concatenate(chunks, axis=0)
mse = float(np.mean((pred - test_y) ** 2))
mse_vars = np.mean((pred - test_y) ** 2, axis=(0, 1))
xc = np.asarray(ds.parameters.xc, dtype=float)
x0 = np.asarray(ds.parameters.x0, dtype=float)
scale = np.array([1.0 / 1e5, 1.0 / 1e5, 3600.0])
pred_eng = (pred[:, 0, :] * xc + x0) * scale
obs_eng = (test_y[:, 0, :] * xc + x0) * scale
err = pred_eng - obs_eng
rmse = np.sqrt(np.mean(err ** 2, axis=0))
mae = np.mean(np.abs(err), axis=0)
names = ["Pbh (bar)", "Pwh (bar)", "q (m3/h)"]
print(f"normalized MSE {mse:.6e}")
for name, part, r, a in zip(names, mse_vars, rmse, mae):
    print(f"{name:12s}  norm MSE {float(part):.6e}  RMSE {float(r):.4f}  MAE {float(a):.4f}")


In [ ]:
from data.forecast import free_run

# 420 feedback steps. free_run also returns the initial one-step prediction.
FORECAST_STEPS = 420
rollout = free_run(
    pinn.u_model,
    ds.un,
    steps=FORECAST_STEPS,
    n_steps_in=ds.n_steps_in,
    parameters=ds.parameters,
    jit_trace=True,
)
yp = rollout.detach().cpu().numpy()[:, 0, :]
phys = (yp * xc + x0) * scale
# Recorded well in engineering units. The rollout starts from the fixed
# initial pressures inside free_run, then writes back only Pbh and Pwh.
measured = np.asarray(ds.x, dtype=float)[: phys.shape[0]] * scale
labels = ["Pbh (bar)", "Pwh (bar)", "q (m3/h)"]
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, name, j in zip(axes, labels, range(3)):
    ax.plot(measured[:, j], "k:", linewidth=1.5, label="measured")
    ax.plot(phys[:, j], label="rollout")
    ax.set_ylabel(name)
    ax.grid(True)
axes[0].legend(loc="upper right")
axes[-1].set_xlabel("Time step")
fig.suptitle(f"Autoregressive rollout, {FORECAST_STEPS} feedback steps")
fig.tight_layout()
plt.show()
print(f"rollout shape {tuple(rollout.shape)}  finite {bool(np.isfinite(phys).all())}")
